In [4]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path

dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

df = pd.read_csv('../windows_ALL2_dac_EPS1_3600_10_unx.csv', index_col=0)
df.drop(columns='mac.1', inplace=True)
cols = [
    'idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac',
    'seconds',
    'qr',
    'num DAC_RANK=1',
    'num DAC_RANK=3',
    'q',
    'nx',
    'qr_p1',
    'q_p1',
    'nx_p1',
    'num DN',
    'avg DN/QR',
    'avg DN/Q',
    'avg DN/NX',
    'std DN/QR',
    'std DN/Q',
    'std DN/NX',
    'max DN/QR',
    'max DN/Q',
    'max DN/NX',
]

df[(df.idxwindow == 2) & ((df.dac_family == 'virut') | (df.dac_family == 'tofsee'))].sort_values(by=['idxwindow', 'mac'])

macs = df.mac.drop_duplicates().to_numpy().tolist()
dac_families = df.dac_family.drop_duplicates().to_numpy().tolist()

# DAC RANK 3 non va usato, sono domini come google.it utilizzati dai malware

df = df[(df.dac_family == 'healthy') | (df["num DAC_RANK=1"] > 0)]

df.shape[0]

924

In [5]:


from typing import Optional, Union


def query_unique_nx(day: int, hour: int, group_mac: bool, dac_family: Optional[str] = None):
    query = f"""
        SELECT
            {'MAC,' if group_mac else 'COUNT(DISTINCT MAC) AS MACS,'}

            COUNT(*) FILTER (WHERE DAC_RANK=1) AS P,
            COUNT(DISTINCT DN_ID) FILTER (WHERE DAC_RANK=1) AS UNIQUE_P,
            COUNT(DISTINCT DN_ID) AS UNIQUE_DN,
            COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3) AS UNIQUE_NXDN,
            COUNT(*) AS QR,
            COUNT(*) filter (WHERE RCODE IS NULL) AS Q,
            COUNT(*) filter (WHERE RCODE IS NOT NULL) AS R,
            COUNT(*) filter (WHERE RCODE=3) AS NX,
            
            COUNT(*) FILTER (WHERE EPS1 >= 0.5) AS P1_QR,
            COUNT(*) FILTER (WHERE RCODE IS NULL AND EPS1 >= 0.5) AS P1_Q,
            COUNT(*) FILTER (WHERE RCODE IS NOT NULL AND EPS1 >= 0.5) AS P1_R,
            COUNT(*) FILTER (WHERE RCODE = 3 AND EPS1 >= 0.5) AS P1_NX,
            COUNT(*) FILTER (WHERE RN=1 AND EPS1 >= 0.5) AS P1_DD,
            COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1 AND EPS1 >= 0.5) AS P1_DD_NX,
            COUNT(*) FILTER (WHERE RN_MAC=1 AND EPS1 >= 0.5) AS P1_DDMAC,
            COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1 AND EPS1 >= 0.5) AS P1_DDMAC_NX,

            SUM(LAMBDA1) AS LLR_QR,
            SUM(LAMBDA1) FILTER (WHERE RCODE IS NULL) AS LLR_Q,
            SUM(LAMBDA1) FILTER (WHERE RCODE IS NOT NULL) AS LLR_R,
            SUM(LAMBDA1) FILTER (WHERE RCODE = 3) AS LLR_NX,
            SUM(LAMBDA1) FILTER (WHERE RN=1) AS LLR_DD,
            SUM(LAMBDA1) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS LLR_DD_NX,
            SUM(LAMBDA1) FILTER (WHERE RN_MAC=1) AS LLR_DDMAC,
            SUM(LAMBDA1) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS LLR_DDMAC_NX

        FROM get_window_all2({day}, 3600, {hour}) 

        WHERE DAC_RANK=3 { f" OR (DAC_RANK=1 AND DAC_FAMILY='{dac_family}')" if dac_family else ""}

        {'GROUP BY MAC' if group_mac else ''}
"""
    with open('./query.log', 'w') as fp:
        fp.write(query)
    return query



In [10]:

def get_df_train(group_mac, day):
    if day < 0 or day > 9: raise Exception(f'Days value is wrong "{day}"')
    ds_train = Path(f'train_groupmac_{group_mac}_{day}.csv')
    if not ds_train.exists():
        rows = []
        for hour in range(24):
            print(hour, 'train', day, f'group mac:{group_mac}', sep='\t')
            row = pd.read_sql(query_unique_nx(day, hour, group_mac, dac_family=None), dbalchemy)
            row.insert(0, 'dac_family', "healthy")
            row.insert(0, 'day', day)
            row.insert(0, 'hour', hour)
            row.insert(0, 'index', hour + day * 24)
            rows.append(row)
            pass
        pd.concat(rows).reset_index(drop=True).to_csv(ds_train)
        pass
    return pd.read_csv(ds_train, index_col=0)

def get_df_test(dac_family, group_mac, day):
    if day <  1 or day > 9: raise Exception(f'Day begin value is wrong "{day}"')
    ds_test = Path(f'test_groupmac_{group_mac}_{dac_family}_{day}.csv')
    if not ds_test.exists():
        rows = []
        for hour in range(24):
            print(hour, 'test', day, f'group mac:{group_mac}', sep='\t')
            row = pd.read_sql(query_unique_nx(day, hour, group_mac, dac_family=dac_family), dbalchemy)
            row.insert(0, 'dac_family', dac_family)
            row.insert(0, 'day', day)
            row.insert(0, 'hour', hour)
            row.insert(0, 'index', hour + day * 24)
            rows.append(row)
            pass
        pd.concat(rows).reset_index(drop=True).to_csv(ds_test)  
        pass
    return pd.read_csv(ds_test, index_col=0)


In [11]:

for group_mac in [False, True]:
    for day in range(10):
        get_df_train(group_mac, day)
        if day > 0:
            for mw in ['virut', 'modpack', 'necurs', 'pitou', 'conficker', 'suppobox', 'tofsee']:
                get_df_test(mw, group_mac, day)
                pass
        pass
    pass


0	test	1	group mac:False
1	test	1	group mac:False
2	test	1	group mac:False
3	test	1	group mac:False
4	test	1	group mac:False
5	test	1	group mac:False
6	test	1	group mac:False
7	test	1	group mac:False
8	test	1	group mac:False
9	test	1	group mac:False
10	test	1	group mac:False
11	test	1	group mac:False
12	test	1	group mac:False
13	test	1	group mac:False
14	test	1	group mac:False
15	test	1	group mac:False
16	test	1	group mac:False
17	test	1	group mac:False
18	test	1	group mac:False
19	test	1	group mac:False
20	test	1	group mac:False
21	test	1	group mac:False
22	test	1	group mac:False
23	test	1	group mac:False
0	test	1	group mac:False
1	test	1	group mac:False
2	test	1	group mac:False
3	test	1	group mac:False
4	test	1	group mac:False
5	test	1	group mac:False
6	test	1	group mac:False
7	test	1	group mac:False
8	test	1	group mac:False
9	test	1	group mac:False
10	test	1	group mac:False
11	test	1	group mac:False
12	test	1	group mac:False
13	test	1	group mac:False
14	test	1	group mac:False
15	tes

In [ ]:
def get_dataset(train_days, dac_family, group_mac):
    df_train = pd.concat([ get_df_train(group_mac, day) for day in range(train_days)])
    df_test  = pd.concat([ get_df_test(dac_family, group_mac, day) for day in range(train_days, 10)])
    return df_train, df_test

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.neighbors import KNeighborsClassifier
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
import matplotlib.pyplot as plt


X = day0_healthy
y = day0_dac_faily
knn = KNeighborsClassifier(n_neighbors=4)

sfs1 = SFS(knn, 
           k_features=3, 
           forward=True, 
           floating=False, 
           verbose=2,
           scoring='accuracy',
           cv=0)

sfs1 = sfs1.fit(X, y)

print('Best accuracy score: %.2f' % sfs1.k_score_)
print('Best subset (indices):', sfs1.k_feature_idx_)
print('Best subset (corresponding names):', sfs1.k_feature_names_)


fig1 = plot_sfs(sfs1.get_metric_dict(), kind='std_dev')

plt.ylim([0.8, 1])
plt.title('Sequential Forward Selection (w. StdDev)')
plt.grid()
plt.show()


[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

for group_mac in [False, True]:
    for train_days in range(1,5):
        for dac_family in ['virut', 'modpack', 'necurs', 'pitou', 'conficker', 'suppobox', 'tofsee']:
            train, test = get_dataset(train_days, dac_family, group_mac)
            cols_values=['qr', 'nx', 'p1_qr', 'p1_nx' ]
            labels = (test['p'] > 0)

            model = IsolationForest(contamination=0.2, random_state=90)
            model.fit(train[cols_values])

            predictions = model.predict(test[cols_values])
            outliers = predictions == -1

            print(dac_family, test['qr'].sum(), end=':')
            print(f'\t\tanormals: {(outliers & labels).sum()} / {labels.sum()}')

            pass


virut 40132916:		anormals: 19 / 19
modpack 40132219:		anormals: 74 / 90
necurs 41785447:		anormals: 63 / 66
pitou 40155825:		anormals: 23 / 33
conficker 40150903:		anormals: 25 / 36
suppobox 40130518:		anormals: 0 / 0
tofsee 40130545:		anormals: 2 / 2


In [88]:
from sklearn.ensemble import IsolationForest
import numpy as np

GROUP_MAC = True
ds_train = Path(f'train_groupmac_{GROUP_MAC}.csv')

if not ds_train.exists():
    day_begin = 0
    day_end = 1
    rows = []
    for day in range(day_begin, day_end):
        for hour in range(24):
            print(day, hour)
            row = pd.read_sql(query_unique_nx(day, hour, GROUP_MAC, dac_family=None), dbalchemy)
            row.insert(0, 'dac_family', "healthy")
            row.insert(0, 'day', day)
            row.insert(0, 'idxdaywindow', hour)
            row.insert(0, 'idxwindow', hour + day * 24)
            rows.append(row)
            pass
        pass
    pd.concat(rows).reset_index(drop=True).to_csv(ds_train)  
    pass

for mw in ['virut', 'modpack', 'necurs', 'pitou', 'conficker', 'suppobox', 'tofsee']:
    df_test = get_df_test(mw, GROUP_MAC, 1)
    pass

for mw in ['virut', 'modpack', 'necurs', 'pitou', 'conficker', 'suppobox', 'tofsee']:
    df_test = get_df_test(mw, GROUP_MAC, 1)

    cols_values=['qr', 'nx', 'p1_qr', 'p1_nx' ]

    train = df_train
    test  = df_test

    labels = (test['p'] > 0)

    print(dac_family, test['qr'].sum(), end=':')

    model = IsolationForest(contamination=0.2, random_state=90)
    model.fit(train[cols_values])

    predictions = model.predict(test[cols_values])
    outliers = predictions == -1

    print(f'\t\tanormals: {(outliers & labels).sum()} / {labels.sum()}')

    pass


0 0
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
0 10
0 11
0 12
0 13
0 14
0 15
0 16
0 17
0 18
0 19
0 20
0 21
0 22
0 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
necurs 40132916:		anormals: 17 / 19
necurs 40132219:		anormals: 81 / 90
necurs 41785447:		anormals: 60 / 66
necurs 40155

In [17]:
emptyrow = [
    0, # seconds
    0, # qr
    0, # num DAC_RANK=1
    0, # num DAC_RANK=3
    0, # q
    0, # nx
    0, # qr_p1
    0, # q_p1
    0, # nx_p1
    0, # num DN
    np.nan, # avg DN/QR
    np.nan, # avg DN/Q
    np.nan, # avg DN/NX
    np.nan, # std DN/QR
    np.nan, # std DN/Q
    np.nan, # std DN/NX
    np.nan, # max DN/QR
    np.nan, # max DN/Q
    np.nan # max DN/NX
]

In [19]:
import pandas as pd

# df = pd.read_csv('windows_dac_EPS1_3600_10.csv', index_col=0)

index_names = ['idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac']
cols_values=['qr', 'num DAC_RANK=1', 'num DAC_RANK=3', 'q', 'nx', 'qr_p1', 'q_p1', 'nx_p1', 'mac2', 'num DN', 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]
df = df.set_index(index_names)

notm = 0
data = [[], []]
for idxday in range(10):
    for idxhour in range(24):
        for idxmw, mw in enumerate(dac_families):
            for idxmac, mac in enumerate(macs):
                idxwindow = idxhour + idxday * 24
                try:
                    df.loc[(idxwindow, idxhour, idxday, mw, mac)]
                    notm+=1
                except KeyError as e:
                    # pd.DataFrame()
                    # missings.append((idxwindow, idxhour, idxday, mw, mac), emptyrow)
                    data[0].append((idxwindow, idxhour, idxday, mw, mac))
                    data[1].append(emptyrow)
                    # df.at[(idxwindow, idxhour, idxday, mw, mac)] = emptyrow
                    pass
                pass
            pass
        pass
    pass

filler = pd.DataFrame(data[1], index=pd.MultiIndex.from_tuples(data[0], names=index_names), columns=cols_values)

df = pd.concat([df, filler]).sort_index().reset_index()

df.shape[0]

/var/folders/4s/h7p_rb6j4ts7ml0z0l5qj_x00000gn/T/ipykernel_85847/1031263206.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, filler]).sort_index().reset_index()


13440

In [32]:
index_names = ['idxwindow', 'idxdaywindow', 'day', 'dac_family', 'mac']
 #, 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]
# cols_values=['nx' ] #, 'avg DN/QR', 'avg DN/Q', 'avg DN/NX', 'std DN/QR', 'std DN/Q', 'std DN/NX', 'max DN/QR', 'max DN/Q', 'max DN/NX' ]

pairs = {}
labels_summary = []
for mac in macs:
    pairs[mac] = {}
    for dac_family in dac_families:
        train = df[(df.mac == mac) & (df.day < 5) & (df.dac_family == 'healthy')]
        test = df[(df.mac == mac) & (df.day >= 5) & (df.dac_family == dac_family)]

        train_label = df[(df.mac == mac) & (df.day < 5) & (df.dac_family == 'healthy')]['num DAC_RANK=1'] > 0
        test_label = df[(df.mac == mac) & (df.day >= 5) & (df.dac_family == dac_family)]['num DAC_RANK=1'] > 0

        pairs[mac][dac_family] = (train, test, train_label, test_label)
        labels_summary.append([mac, dac_family, train_label.sum(), test_label.sum()])
        pass
    pass

labels_summary = pd.DataFrame(labels_summary, columns=['mac', 'mw', 'train', 'test']).set_index(['mac', 'mw'])

74:8e:f8:fb:80:7e virut 53401802:		anormals: 0 / 9
74:8e:f8:fb:80:7e modpack 53402261:		anormals: 27 / 114
74:8e:f8:fb:80:7e necurs 54416008:		anormals: 86 / 107
74:8e:f8:fb:80:7e pitou 53432848:		anormals: 9 / 33
00:e0:20:11:08:e6 virut 59957346:		anormals: 0 / 9
00:e0:20:11:08:e6 modpack 59958787:		anormals: 29 / 115
00:e0:20:11:08:e6 necurs 61174931:		anormals: 99 / 115
00:e0:20:11:08:e6 pitou 59981115:		anormals: 8 / 44
a6:3f:85:2f:c1:e0 virut 31548518:		anormals: 5 / 38
a6:3f:85:2f:c1:e0 modpack 31546569:		anormals: 30 / 115
a6:3f:85:2f:c1:e0 necurs 35676070:		anormals: 101 / 115
a6:3f:85:2f:c1:e0 pitou 31622556:		anormals: 10 / 44
00:04:96:41:28:00 virut 32422665:		anormals: 0 / 1
00:04:96:41:28:00 modpack 32423963:		anormals: 10 / 108
00:04:96:41:28:00 necurs 32599199:		anormals: 47 / 78
00:04:96:41:28:00 pitou 32432650:		anormals: 2 / 17
00:26:cb:32:f2:3f modpack 932429:		anormals: 4 / 17
00:26:cb:32:f2:3f necurs 949916:		anormals: 6 / 6


In [99]:
%pip install tensorflow

import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler

# Creiamo alcuni dati di esempio (traffico normale)
normal_data = np.random.normal(0, 1, (1000, 5))  # Traffico "normale"
anomalous_data = np.random.normal(10, 1, (200, 5))  # Traffico anomalo
test_data = np.vstack([normal_data, anomalous_data])

# Normalizza i dati
scaler = StandardScaler()
normal_data_scaled = scaler.fit_transform(normal_data)
test_data_scaled = scaler.transform(test_data)

# Creiamo un modello di autoencoder
autoencoder = Sequential()
autoencoder.add(Dense(8, activation='relu', input_dim=5))  # Encoder
autoencoder.add(Dense(5, activation='relu'))  # Bottleneck layer
autoencoder.add(Dense(8, activation='relu'))  # Decoder
autoencoder.add(Dense(5, activation='sigmoid'))  # Output layer

autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Alleniamo l'autoencoder sui dati normali
autoencoder.fit(normal_data_scaled, normal_data_scaled, epochs=50, batch_size=32, verbose=1)

# Calcoliamo l'errore di ricostruzione per i dati di test
reconstruction_error = autoencoder.evaluate(test_data_scaled, test_data_scaled)

print(f"Errore di ricostruzione sui dati di test: {reconstruction_error}")

# Una soglia di errore può essere impostata per decidere se un dato è anomalo
threshold = 0.1
predictions = (reconstruction_error > threshold).astype(int)

# Visualizza le anomalie (1: anomalo, 0: normale)
print("Anomalie rilevate:")
print(predictions)


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached wrapt-1.16.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached ml_dtypes-0.4.1-cp312-cp312-macosx_10_9_universal2.whl.metadata (20 kB)
  Using cached Markdown-3.7-py3-none-any.whl.metadata (7.0 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.6/239.6 MB 5.5 MB/s eta 0:00:0000:0100:01
Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
Using cached flatbuffers-24.3.25-py2.py3-

ModuleNotFoundError: No module named 'distutils'

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler

# Creiamo alcuni dati di esempio (traffico normale)
normal_data = np.random.normal(0, 1, (1000, 5))  # Traffico "normale"
anomalous_data = np.random.normal(10, 1, (200, 5))  # Traffico anomalo
test_data = np.vstack([normal_data, anomalous_data])

# Normalizza i dati
scaler = StandardScaler()
normal_data_scaled = scaler.fit_transform(normal_data)
test_data_scaled = scaler.transform(test_data)

# Creiamo un modello di autoencoder
autoencoder = Sequential()
autoencoder.add(Dense(8, activation='relu', input_dim=5))  # Encoder
autoencoder.add(Dense(5, activation='relu'))  # Bottleneck layer
autoencoder.add(Dense(8, activation='relu'))  # Decoder
autoencoder.add(Dense(5, activation='sigmoid'))  # Output layer

autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Alleniamo l'autoencoder sui dati normali
autoencoder.fit(normal_data_scaled, normal_data_scaled, epochs=50, batch_size=32, verbose=1)

# Calcoliamo l'errore di ricostruzione per i dati di test
reconstruction_error = autoencoder.evaluate(test_data_scaled, test_data_scaled)

print(f"Errore di ricostruzione sui dati di test: {reconstruction_error}")

# Una soglia di errore può essere impostata per decidere se un dato è anomalo
threshold = 0.1
predictions = (reconstruction_error > threshold).astype(int)

# Visualizza le anomalie (1: anomalo, 0: normale)
print("Anomalie rilevate:")
print(predictions)


74:8e:f8:fb:80:7e virut 99171403:		anormals: 163 / 14
74:8e:f8:fb:80:7e modpack 99172766:		anormals: 163 / 199
74:8e:f8:fb:80:7e necurs 101375957:		anormals: 194 / 189
74:8e:f8:fb:80:7e pitou 99242148:		anormals: 163 / 73
74:8e:f8:fb:80:7e tofsee 99169632:		anormals: 163 / 1
00:e0:20:11:08:e6 virut 110081080:		anormals: 149 / 16
00:e0:20:11:08:e6 modpack 110084171:		anormals: 149 / 207
00:e0:20:11:08:e6 necurs 112586974:		anormals: 211 / 202
00:e0:20:11:08:e6 pitou 110124684:		anormals: 151 / 86
00:e0:20:11:08:e6 conficker 110091099:		anormals: 148 / 26
00:e0:20:11:08:e6 tofsee 110079477:		anormals: 149 / 1
a6:3f:85:2f:c1:e0 virut 61442648:		anormals: 113 / 72
a6:3f:85:2f:c1:e0 modpack 61439108:		anormals: 113 / 206
a6:3f:85:2f:c1:e0 necurs 70316212:		anormals: 207 / 202
a6:3f:85:2f:c1:e0 pitou 61579493:		anormals: 114 / 86
a6:3f:85:2f:c1:e0 conficker 61470257:		anormals: 113 / 26
a6:3f:85:2f:c1:e0 tofsee 61436958:		anormals: 113 / 1
00:04:96:41:28:00 virut 70045159:		anormals: 168 / 3